# nb04_train : 학습 실험 (E0 스모크 → E1 기준선 → E2 2단계 학습)

- **가속기 : GPU (P100 또는 T4 x1)** · **인터넷 : ON** (첫 실행 시 사전학습 가중치 다운로드)
- **입력** : Dataset `hms-processed`, `hms-src`
- **출력** : `/kaggle/working/models/<EXP>/` (fold별 가중치, OOF 예측, 지표) → 실행 후 Dataset `hms-models`

| 실험 | 입력 | 모델 | 가설 | 실행 설정 |
|---|---|---|---|---|
| **E0** | 10분 스펙트로그램 | EfficientNet-B0 | 파이프라인 점검 | `EXP="E0"` : fold 0, 1 epoch, 512개 |
| **E1** | 10분 스펙트로그램 | EfficientNet-B0 | 기준선 | `EXP="E1"` : 5 fold, 5 epoch, 전체 |
| **E2** | 10분 스펙트로그램 | EfficientNet-B0 | **H1** : 고득표 미세조정 | `EXP="E2"` : 1단계 전체 5 epoch → 2단계 고득표(≥10표) 3 epoch, 학습률 1/10 |

- **E2 판정 방식** : 같은 실행 안에서 1단계(E1과 같은 조건)와 2단계 OOF를 모두 기록해 비교 → 실행 간 GPU 변동을 배제한 통제 비교
- **선택 기준** : E2부터 코드북 5장대로 **고득표 검증 KL**이 가장 낮은 epoch 저장 (E0·E1은 실행 당시 기준인 전체 KL 유지)

**실행 순서** : 새 설정은 `EXP="E0"`로 대화형 점검(약 5분) → 오류 없으면 목표 실험으로 바꿔 Save Version

## 0. 설정 (목적 : 실험 조건 고정 / 출력 : CFG, 경로)

In [1]:
EXP = "E2"            # "E0" : 스모크 / "E1" : 기준선 / "E2" : 2단계 학습 (H1)

import sys, os, json, time, math, copy
from pathlib import Path

def find_root(marker, root="/kaggle/input"):
    for pattern in [f"*/{marker}", f"*/*/{marker}", f"*/*/*/{marker}", f"*/*/*/*/{marker}"]:
        hits = sorted(Path(root).glob(pattern))
        if hits:
            return hits[0].parent
    raise FileNotFoundError(f"{marker} 를 찾지 못했습니다. Add Input 연결을 확인하세요.")

SRC_ROOT = Path(os.environ.get("HMS_SRC_DIR") or find_root("src/data/targets.py").parents[1])
PROC_DIR = Path(os.environ.get("HMS_PROC_DIR") or find_root("eeg50.npy"))
sys.path.insert(0, str(SRC_ROOT))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from src.config import load_config
from src.data.targets import PROB_COLS, CLASSES
from src.data.folds import check_no_leak
from src.eval.metrics import evaluate

CFG = load_config(SRC_ROOT / "configs" / "base.yaml")
SETTINGS = {
    "E0": dict(folds=[0], epochs=1, n_limit=512, select="kl_all", stage2=None),
    "E1": dict(folds=[0, 1, 2, 3, 4], epochs=5, n_limit=None, select="kl_all", stage2=None),
    "E2": dict(folds=[0, 1, 2, 3, 4], epochs=5, n_limit=None, select="kl_high",
               stage2=dict(epochs=3, lr_mult=0.1)),          # 2단계 : 고득표 구간만, 학습률 1/10
}[EXP]
HYPOTHESIS = {"E0": "smoke", "E1": "baseline", "E2": "H1 high-vote fine-tuning"}[EXP]
HP = dict(model="efficientnet_b0", pretrained=True, batch_size=32, lr=1e-3, weight_decay=1e-2,
          num_workers=2, seed=CFG["seed"])

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = Path(os.environ.get("HMS_OUT_DIR", "/kaggle/working")) / "models" / EXP
OUT_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(HP["seed"]); np.random.seed(HP["seed"])
print(f"EXP={EXP} | DEVICE={DEVICE} ({torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU'})")
print("SRC_ROOT :", SRC_ROOT, "\nPROC_DIR :", PROC_DIR, "\nOUT_DIR  :", OUT_DIR)
if DEVICE == "cpu":
    print("[경고] GPU가 꺼져 있습니다. 오른쪽 패널 Accelerator를 GPU로 바꾸세요.")

EXP=E2 | DEVICE=cuda (Tesla T4)
SRC_ROOT : /kaggle/input/datasets/sungik4057/hms-src 
PROC_DIR : /kaggle/input/datasets/sungik4057/hms-processed/hms_processed 
OUT_DIR  : /kaggle/working/models/E2


## 1. 데이터 (목적 : hms-processed 로드·검증 / 출력 : meta, spec 배열)

In [2]:
meta = pd.read_csv(PROC_DIR / "meta.csv")
spec_all = np.load(PROC_DIR / "spec10.npy", mmap_mode="r")
assert len(meta) == spec_all.shape[0], "meta.csv와 배열 행 수가 다릅니다"
assert np.allclose(meta[PROB_COLS].sum(axis=1), 1)
check_no_leak(meta)

if SETTINGS["n_limit"]:
    meta = meta.sample(SETTINGS["n_limit"], random_state=HP["seed"])
meta = meta.reset_index().rename(columns={"index": "row"})   # row : 배열에서의 위치

# 배열을 메모리에 한 번에 올림 : Dataset 디스크에서 무작위로 읽으면 매우 느림 (E0 1 epoch 155초의 원인)
t0 = time.time()
if SETTINGS["n_limit"]:
    rows = np.sort(meta["row"].to_numpy())
    SPEC = np.asarray(spec_all[rows])                          # 필요한 행만 순서대로 읽기
    meta["row"] = np.searchsorted(rows, meta["row"].to_numpy())  # 메모리 배열 기준 위치로 변경
else:
    SPEC = np.load(PROC_DIR / "spec10.npy")                    # 전체 순차 읽기 (약 4GB)
print(f"학습 대상 {len(meta):,}개 | 메모리 적재 {SPEC.shape} {SPEC.dtype} "
      f"{SPEC.nbytes / 1e9:.1f}GB, {time.time() - t0:.0f}s")
display(meta.groupby("fold").agg(n=("eeg_id", "size"), high_vote=("high_vote", "mean")).round(3))

학습 대상 17,089개 | 메모리 적재 (17089, 4, 100, 300) float16 4.1GB, 151s


,n,high_vote
fold,,
0,3345,0.361
1,3389,0.360
2,3461,0.335
3,3277,0.382
4,3617,0.304


In [3]:
class SpecDataset(torch.utils.data.Dataset):
    """10분 스펙트로그램 4부위(LL, RL, LP, RP)를 세로로 이어 붙인 1채널 이미지 (400 x 300)."""
    def __init__(self, df, arr):
        self.rows = df["row"].to_numpy()
        self.y = np.array(df[PROB_COLS].to_numpy(dtype=np.float32), copy=True)   # 쓰기 가능한 복사본 (pandas 읽기 전용 경고 방지)
        self.arr = arr                            # 메모리 배열 (워커는 fork로 복사 없이 공유)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        x = np.asarray(self.arr[self.rows[i]], dtype=np.float32).reshape(-1, 300)   # (400, 300)
        x = (x - x.mean()) / (x.std() + 1e-6)     # 표본별 표준화 : 기록 간 전력 수준 차이 보정
        return torch.from_numpy(x[None]), torch.from_numpy(self.y[i])

ds_check = SpecDataset(meta.head(4), SPEC)
xb, yb = ds_check[0]
print("입력", tuple(xb.shape), "| 타깃 합", float(yb.sum()), "| 평균·표준편차", round(float(xb.mean()), 3), round(float(xb.std()), 3))

입력 (1, 400, 300) | 타깃 합 1.0 | 평균·표준편차 -0.0 1.0


## 2. 모델·학습 함수 (목적 : 동일 조건 학습 루프 / 출력 : train_fold 함수)

In [4]:
def make_model(pretrained):
    try:
        return timm.create_model(HP["model"], pretrained=pretrained, in_chans=1, num_classes=len(CLASSES))
    except Exception as e:
        print(f"[경고] 사전학습 가중치 로드 실패 → 무작위 초기화로 진행 ({type(e).__name__}). 인터넷 설정을 확인하세요.")
        return timm.create_model(HP["model"], pretrained=False, in_chans=1, num_classes=len(CLASSES))

def kl_loss(logits, target):
    """대회 지표와 같은 방향의 손실 : KL(target || softmax(logits))."""
    return F.kl_div(F.log_softmax(logits, dim=1), target, reduction="batchmean")

@torch.no_grad()
def predict(model, loader):
    model.eval(); out = []
    for x, _ in loader:
        with torch.autocast(device_type="cuda", enabled=DEVICE == "cuda"):
            out.append(F.softmax(model(x.to(DEVICE)).float(), dim=1).cpu().numpy())
    return np.concatenate(out)

def make_loader(d, shuffle):
    return torch.utils.data.DataLoader(SpecDataset(d, SPEC), batch_size=HP["batch_size"], shuffle=shuffle,
                                       num_workers=HP["num_workers"], pin_memory=DEVICE == "cuda", drop_last=shuffle)

def run_stage(model, tr, va, epochs, lr, tag, k):
    """한 단계 학습. 검증 fold 전체로 매 epoch 평가하고 SETTINGS['select'] 기준 최적 epoch의 예측·가중치를 반환."""
    dl_tr, dl_va = make_loader(tr, True), make_loader(va, False)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=HP["weight_decay"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=max(epochs * len(dl_tr), 1), pct_start=0.1)
    scaler = torch.amp.GradScaler(enabled=DEVICE == "cuda")
    best, best_pred, best_state = float("inf"), None, None
    for ep in range(epochs):
        model.train(); t0, losses = time.time(), []
        for x, y in dl_tr:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type="cuda", enabled=DEVICE == "cuda"):
                loss = kl_loss(model(x).float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            losses.append(loss.item())
        pred = predict(model, dl_va)
        m = evaluate(va, pred)
        print(f"  fold {k}{(' ' + tag) if tag else ''} ep {ep + 1}/{epochs} | n_train {len(tr):,} | train {np.mean(losses):.4f} | "
              f"val KL {m['kl_all']:.4f} (고득표 {m['kl_high']:.4f}) | {time.time() - t0:.0f}s", flush=True)
        if m[SETTINGS["select"]] < best:           # 선택 기준 최소 epoch 보관
            best, best_pred = m[SETTINGS["select"]], pred
            best_state = copy.deepcopy({kk: v.detach().cpu() for kk, v in model.state_dict().items()})
    return best_pred, best_state

def train_fold(k):
    tr, va = meta[meta.fold != k], meta[meta.fold == k]
    model = make_model(HP["pretrained"]).to(DEVICE)
    pred1, state1 = run_stage(model, tr, va, SETTINGS["epochs"], HP["lr"], "1단계" if SETTINGS["stage2"] else "", k)
    if not SETTINGS["stage2"]:
        torch.save(state1, OUT_DIR / f"fold{k}.pt")
        return va.index, pred1, None
    torch.save(state1, OUT_DIR / f"fold{k}_stage1.pt")
    s2 = SETTINGS["stage2"]
    model.load_state_dict(state1)                  # 1단계 최적 가중치에서 시작
    tr_hv = tr[tr.high_vote == 1]                  # 고득표(≥10표) 구간만으로 미세조정
    pred2, state2 = run_stage(model, tr_hv, va, s2["epochs"], HP["lr"] * s2["lr_mult"], "2단계", k)
    torch.save(state2, OUT_DIR / f"fold{k}.pt")    # 제출용 최종 가중치
    return va.index, pred1, pred2

## 3. 학습 실행 (목적 : fold별 학습·OOF 수집 / 출력 : fold 가중치, oof.csv)

In [5]:
oof = np.full((len(meta), len(CLASSES)), np.nan, dtype=np.float32)
oof_s1 = oof.copy()                                   # E2 : 1단계 OOF (H1 비교용)
t_start = time.time()
for k in SETTINGS["folds"]:
    t0 = time.time()
    idx, pred1, pred2 = train_fold(k)
    oof[idx] = pred2 if pred2 is not None else pred1
    oof_s1[idx] = pred1
    print(f"fold {k} 완료 {(time.time() - t0) / 60:.1f}분 | 누적 {(time.time() - t_start) / 60:.1f}분", flush=True)
gpu_hours = (time.time() - t_start) / 3600
print(f"전체 학습 {gpu_hours * 60:.1f}분 (GPU {gpu_hours:.2f}시간)")

model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

/tmp/ipykernel_23/3954738674.py:38: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()


  fold 0 1단계 ep 1/5 | n_train 13,744 | train 1.0090 | val KL 0.7460 (고득표 0.5789) | 105s
  fold 0 1단계 ep 2/5 | n_train 13,744 | train 0.5446 | val KL 0.7197 (고득표 0.4277) | 90s
  fold 0 1단계 ep 3/5 | n_train 13,744 | train 0.3882 | val KL 0.7103 (고득표 0.4776) | 90s
  fold 0 1단계 ep 4/5 | n_train 13,744 | train 0.2159 | val KL 0.7551 (고득표 0.5568) | 91s
  fold 0 1단계 ep 5/5 | n_train 13,744 | train 0.1286 | val KL 0.7551 (고득표 0.5641) | 91s
  fold 0 2단계 ep 1/3 | n_train 4,732 | train 0.2649 | val KL 0.7078 (고득표 0.3671) | 35s
  fold 0 2단계 ep 2/3 | n_train 4,732 | train 0.1957 | val KL 0.7176 (고득표 0.3762) | 35s
  fold 0 2단계 ep 3/3 | n_train 4,732 | train 0.1611 | val KL 0.7190 (고득표 0.3837) | 35s
fold 0 완료 9.6분 | 누적 9.6분


/tmp/ipykernel_23/3954738674.py:38: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()


...

  fold 2 1단계 ep 1/5 | n_train 13,628 | train 1.0921 | val KL 0.6759 (고득표 0.4885) | 94s
  fold 2 1단계 ep 2/5 | n_train 13,628 | train 0.5492 | val KL 0.6774 (고득표 0.4571) | 90s
  fold 2 1단계 ep 3/5 | n_train 13,628 | train 0.4001 | val KL 0.6479 (고득표 0.4559) | 90s
  fold 2 1단계 ep 4/5 | n_train 13,628 | train 0.2346 | val KL 0.6836 (고득표 0.4796) | 90s
  fold 2 1단계 ep 5/5 | n_train 13,628 | train 0.1395 | val KL 0.6949 (고득표 0.5006) | 90s
  fold 2 2단계 ep 1/3 | n_train 4,780 | train 0.2184 | val KL 0.6570 (고득표 0.3687) | 36s
  fold 2 2단계 ep 2/3 | n_train 4,780 | train 0.1513 | val KL 0.6726 (고득표 0.3845) | 35s
  fold 2 2단계 ep 3/3 | n_train 4,780 | train 0.1237 | val KL 0.6753 (고득표 0.3768) | 35s
fold 2 완료 9.4분 | 누적 28.3분


/tmp/ipykernel_23/3954738674.py:38: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()


  fold 3 1단계 ep 1/5 | n_train 13,812 | train 1.0235 | val KL 0.7704 (고득표 0.5490) | 95s
  fold 3 1단계 ep 2/5 | n_train 13,812 | train 0.5461 | val KL 0.7144 (고득표 0.5207) | 91s
  fold 3 1단계 ep 3/5 | n_train 13,812 | train 0.4005 | val KL 0.7218 (고득표 0.5373) | 91s
  fold 3 1단계 ep 4/5 | n_train 13,812 | train 0.2347 | val KL 0.7571 (고득표 0.5636) | 91s
  fold 3 1단계 ep 5/5 | n_train 13,812 | train 0.1388 | val KL 0.7307 (고득표 0.5507) | 91s
  fold 3 2단계 ep 1/3 | n_train 4,687 | train 0.2737 | val KL 0.6811 (고득표 0.3830) | 35s
  fold 3 2단계 ep 2/3 | n_train 4,687 | train 0.2027 | val KL 0.6985 (고득표 0.3903) | 35s
  fold 3 2단계 ep 3/3 | n_train 4,687 | train 0.1642 | val KL 0.6977 (고득표 0.3962) | 35s
fold 3 완료 9.4분 | 누적 37.7분


/tmp/ipykernel_23/3954738674.py:38: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()


  fold 4 1단계 ep 1/5 | n_train 13,472 | train 1.0091 | val KL 0.7423 (고득표 0.5032) | 95s
  fold 4 1단계 ep 2/5 | n_train 13,472 | train 0.5391 | val KL 0.7554 (고득표 0.4646) | 90s
  fold 4 1단계 ep 3/5 | n_train 13,472 | train 0.3934 | val KL 0.7208 (고득표 0.4903) | 90s
  fold 4 1단계 ep 4/5 | n_train 13,472 | train 0.2299 | val KL 0.7480 (고득표 0.5256) | 90s
  fold 4 1단계 ep 5/5 | n_train 13,472 | train 0.1358 | val KL 0.7734 (고득표 0.5410) | 89s
  fold 4 2단계 ep 1/3 | n_train 4,838 | train 0.2654 | val KL 0.7186 (고득표 0.3729) | 36s
  fold 4 2단계 ep 2/3 | n_train 4,838 | train 0.1969 | val KL 0.7235 (고득표 0.3810) | 36s
  fold 4 2단계 ep 3/3 | n_train 4,838 | train 0.1579 | val KL 0.7359 (고득표 0.3805) | 36s
fold 4 완료 9.4분 | 누적 47.1분
전체 학습 47.1분 (GPU 0.79시간)


## 4. 평가·기록 (목적 : 코드북 지표 산출 / 출력 : metrics.json, results_row.csv)

In [6]:
done = ~np.isnan(oof).any(axis=1)
ev = meta[done]
metrics = evaluate(ev, oof[done], target_spec=CFG["eval"]["target_specificity"])
per_fold = {int(k): round(evaluate(ev[ev.fold == k], oof[done][(ev.fold == k).to_numpy()])["kl_all"], 4)
            for k in SETTINGS["folds"]}

def save_oof(arr, name):
    pd.concat([ev[["eeg_id", "patient_id", "fold", "high_vote"] + PROB_COLS].reset_index(drop=True),
               pd.DataFrame(arr[done], columns=[f"pred_{c}" for c in CLASSES])], axis=1).to_csv(OUT_DIR / name, index=False)
save_oof(oof, "oof.csv")

verdict = None
if SETTINGS["stage2"]:                                 # H1 : 같은 실행 안의 1단계 대비 2단계 개선량
    save_oof(oof_s1, "oof_stage1.csv")
    m1 = evaluate(ev, oof_s1[done], target_spec=CFG["eval"]["target_specificity"])
    delta = m1["kl_high"] - metrics["kl_high"]
    verdict = "지지" if delta >= 0.03 else ("방향 일치·약함" if delta > 0 else "기각")
    cmp = pd.DataFrame({"1단계(전체)": m1, "2단계(고득표 미세조정)": metrics}).loc[
        ["kl_all", "kl_high", "auc_lat", "auc_pd", "sens_at_spec90"]].astype(float).round(4)
    display(cmp)
    print(f"H1 판정 : 고득표 KL 개선 {delta:+.4f} (기준 0.03 이상) → {verdict}")

record = {"exp": EXP, "hypothesis": HYPOTHESIS,
          "change": "Kaggle spectrogram 4x100x300 -> EfficientNet-B0 (in_chans=1)"
                    + (" | stage2: high-vote fine-tune 3ep, lr x0.1" if SETTINGS["stage2"] else ""),
          "cv_kl_all": round(metrics["kl_all"], 4), "cv_kl_high": round(metrics["kl_high"], 4),
          "public_kl": None, "private_kl": None,
          "auc_lat": round(metrics["auc_lat"], 4), "auc_pd": round(metrics["auc_pd"], 4),
          "sens_at_spec90": round(metrics["sens_at_spec90"], 4), "gpu_hours": round(gpu_hours, 2),
          "kaggle_version": None, "verdict": verdict,
          "note": f"per-fold KL {per_fold}" + (f" | stage1 high KL {m1['kl_high']:.4f}" if SETTINGS["stage2"] else "")}
with open(OUT_DIR / "metrics.json", "w") as f:
    json.dump({"record": record, "hp": HP, "settings": SETTINGS, "full": metrics}, f, indent=2, default=float)
pd.DataFrame([record]).to_csv(OUT_DIR / "results_row.csv", index=False)

display(pd.DataFrame([record]).T.rename(columns={0: EXP}))
print("저장 파일 :", sorted(p.name for p in OUT_DIR.iterdir()))

,1단계(전체),2단계(고득표 미세조정)
kl_all,0.6963,0.6865
kl_high,0.4754,0.3847
auc_lat,0.9063,0.9208
auc_pd,0.8963,0.9068
sens_at_spec90,0.8470,0.8962


H1 판정 : 고득표 KL 개선 +0.0907 (기준 0.03 이상) → 지지


,E2
exp,E2
hypothesis,H1 high-vote fine-tuning
change,Kaggle spectrogram 4x100x300 -> EfficientNet-B...
cv_kl_all,0.6865
cv_kl_high,0.3847
public_kl,None
private_kl,None
auc_lat,0.9208
auc_pd,0.9068
sens_at_spec90,0.8962


저장 파일 : ['fold0.pt', 'fold0_stage1.pt', 'fold1.pt', 'fold1_stage1.pt', 'fold2.pt', 'fold2_stage1.pt', 'fold3.pt', 'fold3_stage1.pt', 'fold4.pt', 'fold4_stage1.pt', 'metrics.json', 'oof.csv', 'oof_stage1.csv', 'results_row.csv']


## 다음 단계
- **E0 통과 기준** : 셀 3·4가 오류 없이 끝나고 `cv_kl_all`이 숫자로 출력 (값 자체는 의미 없음)
- **E1 실행** : 셀 0의 `EXP = "E1"`, Accelerator **GPU**, Internet **ON** 확인 → Save Version (예상 1~2시간)
- **E2 실행** : `EXP = "E2"` → Save Version (예상 약 0.8시간 : 1단계 0.6h + 2단계 0.2h)
- **E2 완료 후** : 셀 4의 **1단계·2단계 비교표와 H1 판정** 공유 → 지지 시 nb05에서 `EXP="E2"`로 2차 제출